In [1]:
# Pandas kitabxanasını yükləyirik
import pandas as pd

# Fayl yolunu düzgün göstərmək üçün raw string istifadə edirik
scores = pd.read_csv(r"C:\Users\aysum\m1-07-lab-indexing-aggregation\data_safe_copy.csv")

# İlk 5 sətiri göstəririk
print(scores.head())

# Dataset haqqında qısa məlumat
scores.info()

# 'score' sütununun numeric olduğunu yoxlayırıq
print("Score sütunu dtype:", scores['score'].dtype)

# Sətir sayını yoxlayırıq
print("Number of rows in dataset:", scores.shape[0])

  student_id cohort   module assignment  score
0       S001  alpha  Module1         A1     78
1       S001  alpha  Module1         A2     84
2       S001  alpha  Module2         A1     79
3       S001  alpha  Module2         A2     86
4       S001  alpha  Module3         A1     81
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   student_id  300 non-null    object
 1   cohort      300 non-null    object
 2   module      300 non-null    object
 3   assignment  300 non-null    object
 4   score       300 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 11.8+ KB
Score sütunu dtype: int64
Number of rows in dataset: 300


In [2]:
# Roster DataFrame yaradılır
roster = pd.DataFrame({
    'student_id': ['S001','S002','S003','S004','S005','S200','S201','S202','S203','S204'],
    'status': ['active','active','inactive','active','inactive','active','inactive','active','inactive','active']
})

# Scores ilə left merge edilir
scores_merged = scores.merge(roster, on='student_id', how='left')

# Merge sonrası yoxlanır, statusu olmayanlar
missing_status_count = scores_merged['status'].isna().sum()
print("Number of missing statuses after merge:", missing_status_count)

Number of missing statuses after merge: 270


In [4]:
# Module və cohort üzrə ortalama score
avg_by_module = scores_merged.groupby(['cohort','module'])['score'].mean().reset_index()
print(avg_by_module)

# Cohort üzrə overall ortalama score
avg_by_cohort = scores_merged.groupby('cohort')['score'].mean().reset_index()
print(avg_by_cohort)


  cohort   module      score
0  alpha  Module1  77.970588
1  alpha  Module2  78.382353
2  alpha  Module3  79.794118
3   beta  Module1  78.647059
4   beta  Module2  78.411765
5   beta  Module3  80.029412
6  gamma  Module1  76.218750
7  gamma  Module2  76.468750
8  gamma  Module3  77.843750
  cohort      score
0  alpha  78.715686
1   beta  79.029412
2  gamma  76.843750


In [12]:
# Pivot table: student_id x module
student_module_report = scores_merged.pivot_table(
    index='student_id', 
    columns='module', 
    values='score', 
    aggfunc='mean'
)

# Nəticəni yoxlayırıq
print(student_module_report.head())

# Validation check
assert student_module_report.shape[0] == scores_merged['student_id'].nunique(), "Row count mismatch!"
print("Pivot table rows match unique students.")

module      Module1  Module2  Module3
student_id                           
S001           81.0     82.5     84.5
S002           73.5     72.0     75.0
S003           87.5     88.5     89.5
S004           68.5     67.5     69.5
S005           81.5     82.5     83.5
Pivot table rows match unique students.


In [18]:
# Hər tələbənin cohort üzrə ortalama skorunu hesablayırıq
# Skor sütununa daha mənalı ad veririk
# Hər cohort üzrə tələbələri ortalama skora görə sıralayırıq
# Hər cohort üzrə ilk 3 tələbəni seçirik
# Top_students DataFrame-i sadələşdiririk
# Nəticəni cohort və ortalama skora görə sıralayırıq
# Hər cohort-da 3 tələbə var mı yoxlayırıq
# Nəticəni göstəririk
student_avg = scores.groupby(['student_id', 'cohort'], as_index=False)['score'].mean()
student_avg.rename(columns={'score': 'avg_score'}, inplace=True)

student_avg['rank'] = student_avg.groupby('cohort')['avg_score'].rank(method='first', ascending=False)

top_students = student_avg[student_avg['rank'] <= 3].drop(columns='rank')


top_students = pd.DataFrame({
    'student_id': top_students['student_id'],
    'cohort': top_students['cohort'],
    'avg_score': top_students['avg_score']
})

top_students = top_students.sort_values(
    by=['cohort', 'avg_score'], 
    ascending=[True, False]
).reset_index(drop=True)

cohort_counts = top_students['cohort'].value_counts()
print(f'Validating that all cohorts appear exatly 3 times: {all(cohort_counts == 3)}')

Validating that all cohorts appear exatly 3 times: True
